## Trabajo Práctico: ETL con Python y Pandas

A partir del archivo `ventas.json`, realiza un proceso ETL en Python utilizando Pandas como herramienta principal en sus tres fases. Responde las preguntas propuestas en **celdas Markdown**.

### Fase 1: Extracción

1. Carga `ventas.json` en un DataFrame de Pandas.
    - Presenta el total de registros, los nombres de los campos y los tipos de datos iniciales detectados por Pandas.

2. Realiza un perfil inicial del conjunto de datos utilizando `isna()`, `describe()` y `nunique()`. El perfil debe permitir observar:
    - la cantidad de valores ausentes por campo;
    - un resumen descriptivo de los datos;
    - la cantidad de valores únicos por campo.

A partir de los resultados obtenidos, responde en una celda Markdown: ¿qué problemas podríamos encontrar en el conjunto de datos y qué campos requerirían transformaciones o validaciones?

In [75]:
import pandas as pd

ventas = pd.read_json("ventas.json")
ventas.head()

,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno
0,1,Notebook,Formosa,25000,2.0,pendiente,2026-08-05,NaN
1,2,Notebook,Resistencia,15000,5.0,enviado,2026-08-02,NaN
2,3,Notebook,Formosa,25000,2.0,entregado,2026-08-20,NaN
3,4,Notebook,Resistencia,25000,5.0,enviado,2026-08-08,NaN
4,5,Monitor,Resistencia,35000,1.0,pendiente,2026-08-23,NaN


In [76]:
valores_ausentes= ventas.isna().sum()
valores_ausentes

id                       0
producto                 0
ciudad                   0
precio                   2
cantidad                 1
estado                   1
fecha                    1
comentario_interno    2002
dtype: int64

In [77]:
ventas.describe()

,id,cantidad
count,2003.000000,2002.000000
mean,1000.592611,3.026474
std,577.402809,1.488184
min,1.000000,-10.000000
25%,500.500000,2.000000
50%,1001.000000,3.000000
75%,1500.500000,4.000000
max,2000.000000,5.000000


In [78]:
ventas.nunique(),ventas.count()

(id                    2000
 producto                 5
 ciudad                   5
 precio                  13
 cantidad                11
 estado                   8
 fecha                   36
 comentario_interno       1
 dtype: int64,
 id                    2003
 producto              2003
 ciudad                2003
 precio                2001
 cantidad              2002
 estado                2002
 fecha                 2002
 comentario_interno       1
 dtype: int64)

### Fase 2: Transformación

Realiza las siguientes transformaciones y validaciones sobre el DataFrame utilizando Pandas.

3. Algunas ciudades aparecen escritas con diferencias de mayúsculas, minúsculas o espacios adicionales. Normaliza sus valores para que cada ciudad tenga una única representación.

4. Convierte los precios numéricos almacenados como texto (por ejemplo, `"25000"`) a un tipo numérico adecuado dentro del DataFrame.
    - Identifica como inválidas las ventas cuyos precios no puedan convertirse a un valor numérico.

5. ¿Qué reglas de negocio puedes establecer para los campos `cantidad` y `precio`?
    - Define y valida esas reglas para identificar las ventas inválidas.
    - **Pista**: Revisa el tipo de dato y el **rango** de los campos, con `.max()` y `.min()` de Pandas.

6. Valida mediante Pandas que el campo `estado` contenga uno de los siguientes valores:
    - `pendiente`
    - `enviado`
    - `entregado`

7. Valida que las fechas tengan el formato `YYYY-MM-DD` e identifica las ventas que no cumplan esta condición.

8. ¿Todos los campos son necesarios? Utiliza Pandas para excluir los campos innecesarios del conjunto de datos final.

9. Como última transformación, verifica si existen registros completamente duplicados. Informa cuántos encontraste y elimina las copias adicionales antes de construir el conjunto de datos final.

In [79]:
ventas_copia = ventas.copy()
ventas_copia["ciudad"]= ventas["ciudad"].str.upper().str.strip()

ventas_copia.head()



,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno
0,1,Notebook,FORMOSA,25000,2.0,pendiente,2026-08-05,NaN
1,2,Notebook,RESISTENCIA,15000,5.0,enviado,2026-08-02,NaN
2,3,Notebook,FORMOSA,25000,2.0,entregado,2026-08-20,NaN
3,4,Notebook,RESISTENCIA,25000,5.0,enviado,2026-08-08,NaN
4,5,Monitor,RESISTENCIA,35000,1.0,pendiente,2026-08-23,NaN


In [80]:
ventas["precio"]
pd.to_numeric(ventas["precio"],errors="coerce") 

0        25000.0
1        15000.0
2        25000.0
3        25000.0
4        35000.0
          ...   
1998     15000.0
1999     35000.0
2000    850000.0
2001    120000.0
2002     25000.0
Name: precio, Length: 2003, dtype: float64

In [81]:
ventas["cantidad"].min()


ventas_validas = ventas.loc[ventas["cantidad"] > 0]
ventas = ventas_validas

In [82]:
ventas

,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno
0,1,Notebook,Formosa,25000,2.0,pendiente,2026-08-05,NaN
1,2,Notebook,Resistencia,15000,5.0,enviado,2026-08-02,NaN
2,3,Notebook,Formosa,25000,2.0,entregado,2026-08-20,NaN
3,4,Notebook,Resistencia,25000,5.0,enviado,2026-08-08,NaN
4,5,Monitor,Resistencia,35000,1.0,pendiente,2026-08-23,NaN
...,...,...,...,...,...,...,...,...
1998,1999,Notebook,Resistencia,15000,4.0,entregado,2026-08-06,NaN
1999,2000,Notebook,formosa,35000,2.0,pendiente,2026-08-08,NaN
2000,1257,Mouse,Formosa,850000,3.0,enviado,2026-08-23,NaN
2001,1565,Notebook,Formosa,120000,3.0,entregado,2026-08-24,NaN


In [83]:
ventas.loc[ventas["cantidad"] < 0]

,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno


In [84]:
ventas["cantidad"].max()

np.float64(5.0)

In [85]:
ventas["cantidad"].min()

np.float64(1.0)

In [86]:
ventas["precio"] = pd.to_numeric(ventas["precio"], errors="coerce")
ventas["precio"].min()

np.float64(-25000.0)

In [87]:
ventas_validas = ventas.loc[ventas["precio"] > 0]
ventas = ventas_validas

In [88]:
ventas.loc[ventas["precio"] < 0]

,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno


In [89]:
ventas["precio"].min()

np.float64(15000.0)

## Actividad 6: 

In [90]:
# Ver todos los valores únicos de la columna 'estado'
ventas["estado"].unique()

<StringArray>
[  'pendiente',     'enviado',   'entregado',            '', 'desconocido',
           nan,  'pendiente ',   'cancelado',   'Entregado']
Length: 9, dtype: str

In [91]:
estados_validos = ["pendiente","enviado","entregado"]

ventas = ventas.loc[ventas["estado"].isin(estados_validos)]
ventas["estado"].unique()

<StringArray>
['pendiente', 'enviado', 'entregado']
Length: 3, dtype: str

## Actividad 7 

In [92]:
ventas["fecha"].unique()

<StringArray>
[    '2026-08-05',     '2026-08-02',     '2026-08-20',     '2026-08-08',
     '2026-08-23',     '2026-08-25',     '2026-08-12',     '2026-08-15',
     '2026-08-10',     '2026-08-03',     '2026-08-04',     '2026-08-21',
     '2026-08-13',     '2026-08-07',     '2026-08-24',     '2026-08-19',
     '2026-08-16',     '2026-08-06',     '2026-08-22',     '2026/08/20',
     '2026-08-18',     '2026-08-26',     '2026-08-11',     '2026-08-14',
     '2026-08-17',     '2026-08-01',     '2026-08-27',     '2026-08-09',
      '2026-88-5',     '2026-02-30',     '2026-13-01',     '20-08-2026',
              nan, 'fecha-invalida',               '']
Length: 35, dtype: str

In [99]:
fechas_faltantes = ventas.loc[ventas["fecha"].isna()]
fechas_faltantes

,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno
27,28,Mouse,Resistencia,850000.0,1.0,entregado,NaT,NaN
122,123,Auriculares,Clorinda,35000.0,1.0,entregado,NaT,NaN
227,228,Notebook,Clorinda,120000.0,3.0,entregado,NaT,NaN
319,320,Auriculares,FORMOSA,25000.0,4.0,entregado,NaT,NaN
587,588,Teclado,formosa,120000.0,5.0,enviado,NaT,NaN
1315,1316,Mouse,Resistencia,15000.0,3.0,enviado,NaT,NaN
1423,1424,Mouse,FORMOSA,850000.0,2.0,enviado,NaT,NaN
1804,1805,Monitor,FORMOSA,850000.0,4.0,enviado,NaT,NaN


In [93]:
ventas["fecha"] = pd.to_datetime(ventas["fecha"], errors="coerce")
ventas


,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno
0,1,Notebook,Formosa,25000.0,2.0,pendiente,2026-08-05,NaN
1,2,Notebook,Resistencia,15000.0,5.0,enviado,2026-08-02,NaN
2,3,Notebook,Formosa,25000.0,2.0,entregado,2026-08-20,NaN
3,4,Notebook,Resistencia,25000.0,5.0,enviado,2026-08-08,NaN
4,5,Monitor,Resistencia,35000.0,1.0,pendiente,2026-08-23,NaN
...,...,...,...,...,...,...,...,...
1998,1999,Notebook,Resistencia,15000.0,4.0,entregado,2026-08-06,NaN
1999,2000,Notebook,formosa,35000.0,2.0,pendiente,2026-08-08,NaN
2000,1257,Mouse,Formosa,850000.0,3.0,enviado,2026-08-23,NaN
2001,1565,Notebook,Formosa,120000.0,3.0,entregado,2026-08-24,NaN


In [94]:
ventas["fecha"].unique()

<DatetimeArray>
['2026-08-05 00:00:00', '2026-08-02 00:00:00', '2026-08-20 00:00:00',
 '2026-08-08 00:00:00', '2026-08-23 00:00:00', '2026-08-25 00:00:00',
 '2026-08-12 00:00:00', '2026-08-15 00:00:00', '2026-08-10 00:00:00',
 '2026-08-03 00:00:00', '2026-08-04 00:00:00', '2026-08-21 00:00:00',
 '2026-08-13 00:00:00', '2026-08-07 00:00:00', '2026-08-24 00:00:00',
 '2026-08-19 00:00:00', '2026-08-16 00:00:00', '2026-08-06 00:00:00',
 '2026-08-22 00:00:00',                 'NaT', '2026-08-18 00:00:00',
 '2026-08-26 00:00:00', '2026-08-11 00:00:00', '2026-08-14 00:00:00',
 '2026-08-17 00:00:00', '2026-08-01 00:00:00', '2026-08-27 00:00:00',
 '2026-08-09 00:00:00']
Length: 28, dtype: datetime64[us]

## Actividad 8 :

In [102]:
ventas.columns

Index(['id', 'producto', 'ciudad', 'precio', 'cantidad', 'estado', 'fecha',
       'comentario_interno'],
      dtype='str')

In [105]:
ventas = ventas.drop(columns=["comentario_interno"])
ventas

,id,producto,ciudad,precio,cantidad,estado,fecha
0,1,Notebook,Formosa,25000.0,2.0,pendiente,2026-08-05
1,2,Notebook,Resistencia,15000.0,5.0,enviado,2026-08-02
2,3,Notebook,Formosa,25000.0,2.0,entregado,2026-08-20
3,4,Notebook,Resistencia,25000.0,5.0,enviado,2026-08-08
4,5,Monitor,Resistencia,35000.0,1.0,pendiente,2026-08-23
...,...,...,...,...,...,...,...
1998,1999,Notebook,Resistencia,15000.0,4.0,entregado,2026-08-06
1999,2000,Notebook,formosa,35000.0,2.0,pendiente,2026-08-08
2000,1257,Mouse,Formosa,850000.0,3.0,enviado,2026-08-23
2001,1565,Notebook,Formosa,120000.0,3.0,entregado,2026-08-24


In [106]:
ventas.columns

Index(['id', 'producto', 'ciudad', 'precio', 'cantidad', 'estado', 'fecha'], dtype='str')

## Actividad 9:

In [ ]:
registros_duplicados= ventas.duplicated().sum()
registros_duplicados

np.int64(3)

In [111]:
ventas = ventas.drop_duplicates()
ventas

,id,producto,ciudad,precio,cantidad,estado,fecha
0,1,Notebook,Formosa,25000.0,2.0,pendiente,2026-08-05
1,2,Notebook,Resistencia,15000.0,5.0,enviado,2026-08-02
2,3,Notebook,Formosa,25000.0,2.0,entregado,2026-08-20
3,4,Notebook,Resistencia,25000.0,5.0,enviado,2026-08-08
4,5,Monitor,Resistencia,35000.0,1.0,pendiente,2026-08-23
...,...,...,...,...,...,...,...
1995,1996,Mouse,Resistencia,120000.0,2.0,pendiente,2026-08-27
1996,1997,Mouse,Formosa,25000.0,1.0,entregado,2026-08-23
1997,1998,Mouse,Clorinda,15000.0,4.0,enviado,2026-08-02
1998,1999,Notebook,Resistencia,15000.0,4.0,entregado,2026-08-06


## Fase 3: Carga

A partir de los resultados obtenidos con Pandas:

10. Construye el DataFrame final de ventas **válidas y transformadas**. Toda venta que incumpla al menos una validación debe quedar excluida de este DataFrame.

11. Antes de guardar los resultados, presenta un resumen de calidad que incluya:

- cantidad total de ventas procesadas;
- cantidad de registros duplicados eliminados;
- cantidad de ventas válidas;
- cantidad de ventas rechazadas.

12. Guarda el DataFrame de ventas válidas en un nuevo archivo llamado `ventas_limpias.json`.

## Desafío opcional: registro de *dead letters*

Además del proceso ETL principal, puedes conservar los errores detectados en un DataFrame separado. Cada fila debe representar un único error e incluir los datos originales completos de la venta, el campo inválido, su valor original y la razón del error. Si una venta presenta más de un error, debe generar una fila por cada validación incumplida.

Por ejemplo, una fila del DataFrame de errores podría tener la siguiente estructura:

```json
{
    "id": 101,
    "producto": "Mouse",
    "ciudad": "Formosa",
    "precio": -15000,
    "cantidad": 2,
    "estado": "pendiente",
    "fecha": "2026-08-12",
    "comentario_interno": "dato que no necesitamos",
    "campo_invalido": "precio",
    "valor_original": -15000,
    "razon": "el precio debe ser mayor que cero"
}
```

Amplía el resumen de calidad para incluir:

- cantidad total de errores;
- cantidad de errores por campo y por razón.

Explica en una celda Markdown por qué la cantidad de ventas rechazadas puede ser diferente de la cantidad total de errores. Finalmente, guarda el DataFrame de errores en un archivo llamado `errors.json`.

**Dato curioso:** en los procesos de datos, los registros que no pueden continuar por errores de validación suelen denominarse *dead letters*. Separarlos permite analizarlos y corregirlos sin detener el procesamiento de los registros válidos.